<a href="https://colab.research.google.com/github/Adrien-Rouillon/Machine-learning---SpotifyDB/blob/main/Notebooks/Delivrable2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MOD10 – Project, Deliverable 2 : exploration and pre-processing data
**Membres du groupe :** Adrien ROUILLON, Paul MORAND

**Data set :** Spotify

---

In [1]:
from google.colab import files
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
url = "https://raw.githubusercontent.com/Adrien-Rouillon/Machine-learning---SpotifyDB/refs/heads/main/spotify_artist_streaming_2020_2025.csv?token=GHSAT0AAAAAAEJEI4BTGIKLA4WVR6STALXI2WZGLAA"
df = pd.read_csv(url)


Before we need to do any data analysis, we need to mess it up since we chose a perfect dataset.

We gave the prompt from class 2 notebook into Claude. It generated us a "add_noise" function to make the data less clean.

In [5]:
def add_noise(df, seed=42):  # return a messy copy of a clean DataFrame
    rng = np.random.default_rng(seed)  # fixed seed -> same mess every run
    out = df.copy(deep=True)  # work on a deep copy so df is never modified
    num_cols = out.select_dtypes("number").columns.tolist()  # numeric columns available to corrupt
    if len(num_cols) < 2:  # need at least two numeric columns
        raise ValueError("df needs at least 2 numeric columns")  # fail loudly instead of guessing
    miss_cols = num_cols[:2]  # first two numeric columns get missing values
    out_col = num_cols[0]  # first numeric column gets outliers
    text_col = num_cols[2] if len(num_cols) > 2 else num_cols[1]  # third (or second) becomes text
    n = len(out)  # number of original rows

    # --- extreme outliers ---
    out_idx = rng.choice(out.index, size=3, replace=False)  # pick 3 rows to corrupt
    col = out[out_col].astype(float)  # float copy so large values fit
    col.loc[out_idx] = col.mean() + 20 * col.std() * rng.choice([-1, 1], size=3)  # push ±20 std from the mean
    out[out_col] = col  # write outliers back

    # --- ~5% missing values in two numeric columns ---
    for c in miss_cols:  # handle each target column
        nan_idx = rng.choice(out.index, size=max(1, round(0.05 * n)), replace=False)  # 5% random rows
        out[c] = out[c].astype(float)  # ints can't hold NaN, so cast to float
        out.loc[nan_idx, c] = np.nan  # blank them out

    # --- inconsistent categorical column ---
    base = rng.choice(["Red", "Green", "Blue"], size=n)  # clean underlying categories
    variants = [str.lower, str.upper, str.title, lambda s: f" {s}", lambda s: f"{s}  "]  # case/spacing styles
    out["color"] = [variants[rng.integers(len(variants))](v) for v in base]  # apply a random style per row

    # --- numeric column stored as text with a stray 'unknown' ---
    out[text_col] = out[text_col].map(lambda v: "" if pd.isna(v) else str(v)).astype(object)  # numbers -> strings
    out.loc[rng.choice(out.index), text_col] = "unknown"  # one non-numeric token

    # --- 5 exact duplicate rows ---
    dup_rows = out.loc[rng.choice(out.index, size=5, replace=False)]  # copy 5 existing (already messy) rows
    out = pd.concat([out, dup_rows], ignore_index=True)  # append them -> exact duplicates
    out = out.sample(frac=1, random_state=seed).reset_index(drop=True)  # shuffle so dupes aren't at the end

    return out  # the messy copy

df = add_noise(df)

## Étape 1 – Exploration (EDA)



---



In [3]:
df.head()

,track_id,track_name,artist_name,album_name,release_date,genre,duration_ms,popularity,danceability,energy,...,popularity_category,loudness_category,key_name,mode_name,is_explicit_bool,release_quarter,is_weekend_release,log_stream_count,upbeat_score,artist_track_count
0,346d85e2-ffde-41b3-854a-589ed880b91f,Storm,Ethereal,The Hollow Journeys,2021-10-06,Pop,169594,47,0.6777,0.7949,...,Medium,Loud,A,Major,False,Q4,False,10.7492,0.6632,3232
1,12bc268a-363b-4f19-8fa8-0ced7ae1e462,Silent,The Wolfs,Ancient Thunder,2024-10-26,K-Pop,256755,26,0.9405,0.7163,...,Medium,Moderate,C#,Major,False,Q4,True,8.5895,0.7615,20
2,30d627d9-878d-4bb3-be59-71feb6ed4f3f,Shattered Lines,LolaOre,Twisted Harbors,2022-11-12,Punk,170212,46,0.6853,0.9500,...,Medium,Loud,E,Major,False,Q4,True,12.2935,0.7828,199
3,85501bb9-d221-4339-9e2e-29b1b3daff0a,Breaking Roses,Clay Black,The Blinding Letters,2020-06-22,Pop,181250,4,0.7817,0.6943,...,Low,Moderate,F#,Major,False,Q2,False,8.5658,0.6778,693
4,7bf92bd2-c6fe-4869-a90e-242bb8bbe7a3,Lines,Knot Hart,The Cosmic Letters,2024-01-22,Latin,160894,27,0.7200,0.4958,...,Medium,Moderate,C#,Major,False,Q1,False,10.2191,0.5494,12111


In [ ]:
df.shape

(50000, 33)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 33 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   track_id             50000 non-null  object 
 1   track_name           50000 non-null  object 
 2   artist_name          50000 non-null  object 
 3   album_name           50000 non-null  object 
 4   release_date         50000 non-null  object 
 5   genre                50000 non-null  object 
 6   duration_ms          50000 non-null  int64  
 7   popularity           50000 non-null  int64  
 8   danceability         50000 non-null  float64
 9   energy               50000 non-null  float64
 10  key                  50000 non-null  int64  
 11  loudness             50000 non-null  float64
 12  mode                 50000 non-null  int64  
 13  instrumentalness     50000 non-null  float64
 14  tempo                50000 non-null  float64
 15  stream_count         50000 non-null 

In [ ]:
df.describe()

,duration_ms,popularity,danceability,energy,key,loudness,mode,instrumentalness,tempo,stream_count,release_year,release_month,duration_minutes,log_stream_count,upbeat_score,artist_track_count
count,50000.000000,50000.000000,50000.000000,50000.000000,50000.00000,50000.000000,50000.000000,50000.000000,50000.000000,5.000000e+04,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000
mean,209814.455560,28.136820,0.661236,0.631853,5.47086,-7.669547,0.650280,0.055166,119.067461,1.183789e+05,2022.499620,6.528880,3.496915,9.835507,0.601615,4052.066320
std,44646.016547,15.999501,0.164143,0.168741,3.45610,2.621917,0.476886,0.095083,19.647807,5.658443e+05,1.709953,3.460926,0.744101,1.793877,0.105076,4817.780532
min,90000.000000,0.000000,0.000000,0.000000,0.00000,-18.620000,0.000000,0.000000,60.000000,1.000000e+02,2020.000000,1.000000,1.500000,4.615100,0.057800,2.000000
25%,179753.000000,16.000000,0.562400,0.523400,2.00000,-9.440000,0.000000,0.010300,105.990000,4.979750e+03,2021.000000,4.000000,3.000000,8.513350,0.542100,181.000000
50%,209689.000000,26.000000,0.675000,0.633400,5.00000,-7.660000,1.000000,0.025700,118.575000,1.595500e+04,2023.000000,7.000000,3.490000,9.677600,0.606400,1486.000000
75%,239708.250000,39.000000,0.775400,0.744700,8.00000,-5.880000,1.000000,0.057600,131.350000,6.068550e+04,2024.000000,10.000000,4.000000,11.013525,0.671100,5271.000000
max,388117.000000,92.000000,1.000000,1.000000,11.00000,0.000000,1.000000,1.000000,220.000000,4.334502e+07,2025.000000,12.000000,6.470000,17.584700,0.915400,12111.000000




---



**Observation:**

.head() : shows us the first five rows of the data base.

.shape : give us the size of the data base (rows, columns)

## Étape 2 – Missing values



---



In [6]:
df.isna().sum()

,0
track_id,0
track_name,0
artist_name,0
album_name,0
release_date,0
genre,0
duration_ms,2500
popularity,2500
danceability,0
energy,0


**Observation:**

.isna() : Returns us the table with either True or False depending if the cell is empty or not.

.sum() : Counts each True in a column.

## Étape 3 – Encoding



---



In [ ]:
df["explicit"] = df["explicit"].map({"True" : 1, "False" : 0})
df["mode_name"] = df["mode_name"].map({"Major" : 1, "Minor" : 0})
df["artist_upper_name"] = df["artist_name"].map(str.upper)

In [ ]:
df = pd.get_dummies(df, columns=["genre"])
df = pd.get_dummies(df, columns=["country"])

KeyError: "None of [Index(['country'], dtype='object')] are in the [columns]"

## Étape 4 – Scaling



---

